# Cell marker positivity — inference on OME-TIFF

The same as `Inference.ipynb`, but reading the marker stack and the segmentation straight
from OME-TIFF instead of NIfTI. The marker stack can optionally be written out as a NIfTI
z-stack, which is the form `TrainModel.ipynb` expects.

In [ ]:
import numpy as np
import nibabel as nib
import tensorflow as tf
import tifffile
import matplotlib.pyplot as plt
from scipy import ndimage

MARKERS_TIFF = "TrainData/20231201_713Lung_E_3_20231201-154853-900840_Q013.ome.tiff"
MASK_TIFF = "TrainData/20231201_713Lung_E_3_20231201-154853-900840_Q013_mask.ome.tiff"
OUT_PATH = "TrainData/20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_positivity_predicted.nii.gz"
MODEL_PATH = "marker_cnn_final.h5"

# The marker stack as a NIfTI, in the form TrainModel.ipynb reads. Off by default so a
# run cannot overwrite a stack that was already used for training.
SAVE_MARKER_STACK = False
MARKER_STACK_PATH = "TrainData/20231201_713Lung_E_3_20231201-154853-900840_Q013_markers_channels_zstack.nii.gz"

CROP = 64
PERCENTILE = 99        # must match TrainModel.ipynb
THRESHOLD = 0.5        # probability above which a cell is called positive
BATCH_SIZE = 512
CHUNK = 2048           # cells prepared at a time
DETAIL_CHANNEL = 0     # channel to show close up at the end

print("TensorFlow:", tf.__version__)

# Only the network is needed, not the optimizer it was trained with. Leaving that out
# also lets a checkpoint from a newer TensorFlow load in an older one, such as the
# TF 2.8 of Opal Studio.
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

# The checkpoint records how it was trained. The marker channels are an odd number, so
# the channel count alone says whether the mask of the other cells is there as well.
N_CHANNELS = model.input_shape[-1]
USE_OTHER_CELLS = N_CHANNELS % 2 == 1
N_MARKERS = N_CHANNELS - 1 - int(USE_OTHER_CELLS)
N_SIDE = (N_MARKERS - 1) // 2

print(f"Model expects {N_CHANNELS} channels: {N_MARKERS} marker channels "
      f"({N_SIDE} on each side of the target)"
      f"{', the target cell and the other cells' if USE_OTHER_CELLS else ' and the target cell'}")

## Load the region

The OME-TIFF axes are reordered to `(height, width, channels)`; a leading time axis is
dropped. The segmentation must be a single image, which is binarised and then split into
cells by a connected component analysis, exactly as during training.

In [ ]:
def load_ome_tiff(path):
    """Read an OME-TIFF as (height, width, channels), first timepoint only."""
    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        data = series.asarray()
        axes = series.axes

    if "T" in axes:
        data = np.take(data, 0, axis=axes.index("T"))
        axes = axes.replace("T", "")

    if "Y" not in axes or "X" not in axes:
        raise ValueError(f"no Y and X axes in '{axes}'")

    order = [axes.index("Y"), axes.index("X")]
    order += [i for i in range(len(axes)) if i not in order]

    data = np.transpose(data, order)
    print(f"  {path.split('/')[-1]}: {series.shape} '{axes}' -> "
          f"{data.shape[0]}x{data.shape[1]} with {int(np.prod(data.shape[2:]))} channels")

    return np.ascontiguousarray(data).reshape(data.shape[0], data.shape[1], -1)


markers = load_ome_tiff(MARKERS_TIFF).astype(np.float32)

mask = load_ome_tiff(MASK_TIFF)
if mask.shape[2] != 1:
    raise ValueError(f"the segmentation must be a single image, got {mask.shape[2]}")
mask = mask[:, :, 0]

if mask.shape != markers.shape[:2]:
    raise ValueError(f"mask is {mask.shape}, markers are {markers.shape[:2]}")

cells, n_cells = ndimage.label(mask > 0)
cells = cells.astype(np.int32)
centres = np.array(ndimage.center_of_mass(
    cells > 0, cells, np.arange(1, n_cells + 1))).astype(np.int32)

print(f"{markers.shape[2]} channels, {n_cells} cells")

if SAVE_MARKER_STACK:
    nib.save(nib.Nifti1Image(markers, np.eye(4)), MARKER_STACK_PATH)
    print(f"wrote {MARKER_STACK_PATH}")

## Running the model

`make_sample` cuts exactly the same crop as `TrainModel.ipynb`: the target channel with
`N_SIDE` neighbours on each side, empty where the stack runs out, each normalised on its
own by its `PERCENTILE` within the crop and clipped at 1, then the mask of the target
cell and, when the model was trained with it, the mask of the other cells.
No augmentation and no simulated spillover are applied here.

In [ ]:
def normalise(planes):
    """Scale every marker channel of a crop by its own percentile and clip it at 1."""
    scale = np.percentile(planes, PERCENTILE, axis=(0, 1))

    # A sparse channel can have a zero percentile and still carry a little signal, which
    # its maximum then scales; a channel that is empty throughout stays empty.
    scale = np.where(scale > 0, scale, planes.max(axis=(0, 1)))
    scale[scale == 0] = 1.0

    return np.minimum(planes / scale, 1.0).astype(np.float32)


def make_sample(z, cell_id, centre_y, centre_x):
    """The same crop the model was trained on."""
    height, width, n_z = markers.shape
    half = CROP // 2

    y = min(max(centre_y - half, 0), height - CROP)
    x = min(max(centre_x - half, 0), width - CROP)

    # Channels beyond either end of the stack do not exist and stay empty.
    wanted = np.arange(z - N_SIDE, z + N_SIDE + 1)
    exists = (wanted >= 0) & (wanted < n_z)

    planes = np.zeros((CROP, CROP, N_MARKERS), dtype=np.float32)
    planes[..., exists] = markers[y:y + CROP, x:x + CROP, wanted[exists]]

    crop = cells[y:y + CROP, x:x + CROP]
    stack = [normalise(planes), (crop == cell_id)[..., None]]
    if USE_OTHER_CELLS:
        stack.append(((crop > 0) & (crop != cell_id))[..., None])

    return np.concatenate(stack, axis=-1).astype(np.float32)


n_z = markers.shape[2]
positivity = np.zeros(markers.shape, dtype=np.uint8)
probability = np.zeros((n_cells, n_z), dtype=np.float32)

for z in range(n_z):
    for start in range(0, n_cells, CHUNK):
        batch_ids = np.arange(start, min(start + CHUNK, n_cells))
        batch = np.stack([make_sample(z, i + 1, *centres[i]) for i in batch_ids])
        probability[batch_ids, z] = model.predict(
            batch, batch_size=BATCH_SIZE, verbose=0).ravel()

    # 0 stays background, 1 is negative and 2 is positive, as in the training masks
    call = np.concatenate([[0], np.where(probability[:, z] >= THRESHOLD, 2, 1)])
    positivity[:, :, z] = call[cells]

    print(f"  channel {z + 1:3d}/{n_z}: {(probability[:, z] >= THRESHOLD).mean():6.1%} positive")

print("\nDone.")

## Save

In [ ]:
nib.save(nib.Nifti1Image(positivity, np.eye(4)), OUT_PATH)
print(f"wrote {OUT_PATH}  {positivity.shape}  {positivity.dtype}")

## Results

In [ ]:
columns = 8
rows = int(np.ceil(n_z / columns))

fig, axes = plt.subplots(rows, columns, figsize=(2.0 * columns, 2.2 * rows))
for z, ax in enumerate(np.atleast_1d(axes).ravel()):
    ax.axis("off")
    if z < n_z:
        ax.imshow(positivity[:, :, z], vmin=0, vmax=2, cmap="viridis", interpolation="nearest")
        ax.set_title(f"z{z}   {(probability[:, z] >= THRESHOLD).mean():.0%}+", fontsize=8)
fig.suptitle("Predicted positivity per channel (dark = negative, bright = positive)")
plt.tight_layout()
plt.show()

z = DETAIL_CHANNEL
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(markers[:, :, z], cmap="gray",
               vmax=max(np.percentile(markers[:, :, z], 99.5), 1e-6))
axes[0].set_title(f"marker channel {z}")
axes[1].imshow(cells > 0, cmap="gray")
axes[1].set_title(f"cell segmentation ({n_cells} cells)")
axes[2].imshow(positivity[:, :, z], vmin=0, vmax=2, cmap="viridis", interpolation="nearest")
axes[2].set_title(f"prediction ({(probability[:, z] >= THRESHOLD).mean():.1%} positive)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()